# Great Dane Analysis

Sections:
1. **Load & clean**: reads the Airtable grid exports and standardizes types and text
2. **Data-quality checks**: flags anything that doesn't reconcile, looks like test data, or is missing
3. **Analysis**: one section per question, all built on the cleaned frames

In [ ]:
import csv
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

AS_OF = pd.Timestamp('2026-10-07')   # export date of the grid views
FILES = {
    'dealers': 'Dealers_Grid view_2026-10-07.xlsx',
    'budgets': 'Dealer & Budgets_Grid view_2026-10-07.xlsx',
    'teams':   'Dealer_Teams_Grid view_2026-10-07.xlsx',
    'opps':    'Salesforce Opportunities_Grid view_2026-10-07.xlsx',
}
key = 'Dealer Name'   # join key across the dealer-level grids

## 1. Load & clean

Cleaning rules applied to every export:
- Trim stray leading/trailing/double whitespace in text, e.g. `' Great Dane Columbus'`, `'Randy Gosson '`, `'Corporate-Owned Branch '`
- Blank or whitespace-only strings become `NaN`
- `$1,234.00` becomes a float; `59%` becomes 0.59; `✅` / `checked` / blank become a boolean
- Date columns are parsed; columns that are 100% empty are dropped (and logged)
- Multi-value link fields (comma-separated) are left as text but can be exploded with `split_list()`

In [ ]:
cleaning_log = []

def log(frame, step, detail):
    cleaning_log.append({'frame': frame, 'step': step, 'detail': detail})

def to_money(s):
    return pd.to_numeric(s.astype('string').str.replace(r'[$,\s]', '', regex=True), errors='coerce')

def to_pct(s):
    return pd.to_numeric(s.astype('string').str.replace('%', '', regex=False), errors='coerce') / 100

def to_flag(s):
    return s.astype('string').str.strip().isin(['✅', 'checked', 'True', 'true', 'Yes', '1'])

def spearman(a, b):
    """Rank correlation without needing scipy."""
    return a.rank().corr(b.rank())

def split_list(s):
    """Split a comma-separated Airtable link field into a list of trimmed values.
    Airtable wraps names that contain commas in double quotes, so parse it as CSV."""
    def parse(v):
        if pd.isna(v) or not str(v).strip():
            return []
        row = next(csv.reader([str(v)], skipinitialspace=True))
        return [x.strip() for x in row if x.strip()]
    return s.apply(parse)

def text_cols(df):
    # object (pandas 2) or str (pandas 3) columns that hold text
    return [c for c in df.columns
            if df[c].dtype == object or pd.api.types.is_string_dtype(df[c].dtype)]

def norm_col(c):
    return re.sub(r'\s+', ' ', c).strip()

def clean_frame(df, name, money=(), pct=(), flags=(), dates=(), drop=()):
    df = df.copy()
    df.columns = [norm_col(c) for c in df.columns]
    money, pct, flags, dates, drop = ([norm_col(c) for c in cols] for cols in (money, pct, flags, dates, drop))

    # Text: trim whitespace, collapse internal runs, blank -> NaN
    for c in text_cols(df):
        raw = df[c]
        s = raw.astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
        s = s.mask(s == '')
        changed = (raw.notna() & (raw.astype('string') != s)).sum()
        if changed:
            log(name, 'trimmed whitespace / blanked', f'{c}: {changed} values')
        df[c] = s.astype(object).where(s.notna(), np.nan)

    for c in money: df[c] = to_money(df[c])
    for c in pct:   df[c] = to_pct(df[c])
    for c in flags: df[c] = to_flag(df[c])
    for c in dates: df[c] = pd.to_datetime(df[c], errors='coerce')

    empty = [c for c in df.columns if df[c].isna().all()]
    if empty:
        log(name, 'dropped empty columns', ', '.join(empty))
    to_drop = empty + [c for c in drop if c in df.columns]
    for c in drop:
        if c in df.columns:
            log(name, 'dropped redundant column', c)
    return df.drop(columns=to_drop)

money_cols = lambda df: [c for c in df.columns
                         if df[c].astype('string').str.fullmatch(r'-?\$[\d,]+(\.\d+)?').fillna(False).any()]
pct_cols   = lambda df: [c for c in df.columns
                         if df[c].astype('string').str.fullmatch(r'-?\d+(\.\d+)?%').fillna(False).any()]

raw = {k: pd.read_excel(v) for k, v in FILES.items()}
for k, df in raw.items():
    print(f'{k:8} {df.shape}')

In [ ]:
dealers = clean_frame(
    raw['dealers'], 'dealers',
    money=money_cols(raw['dealers']), pct=pct_cols(raw['dealers']),
    flags=['Active'], dates=['Join Date'],
    drop=['Promo Wallet Usage % copy',   # rounded 0/1 duplicate of Promo Wallet Usage %
          'Days Since Joined'],          # text like '48years'; recomputed below
)
dealers['Years Since Joined'] = ((AS_OF - dealers['Join Date']).dt.days / 365.25).round(1)
dealers = dealers.rename(columns={
    'Marketing Fund Budget Rollup (from Reimbursement Dealer Summary)': 'Marketing Fund Budget'})

budgets = clean_frame(raw['budgets'], 'budgets',
                      money=money_cols(raw['budgets']), pct=pct_cols(raw['budgets']))

teams = clean_frame(raw['teams'], 'teams', pct=pct_cols(raw['teams']))

opps = clean_frame(
    raw['opps'], 'opps',
    money=['Opportunity Amount'], flags=['Pitched? (any)', 'Pitched & Opened?'],
    dates=['Created Date', 'Close Date', 'First Opened Pitch'],
)
opps['Counts as influenced?'] = opps['Counts as influenced?'].astype(bool)
opps['Is Closed'] = opps['Opportunity Stage'].str.startswith('Closed')
opps['Is Won'] = opps['Opportunity Stage'].eq('Closed Won')

# Dealer/Channel can hold several linked dealers ('National Accounts, Cox Automotive')
opps['Dealer List'] = split_list(opps['Dealer/Channel'])

pd.DataFrame(cleaning_log)

### 1b. Reference data: dealer tiers & size

From `2026_RMF_Budget_Dealer_Tiers.xlsx`:
- **FINAL** sheet: dealer score and **tier (1–4)**
- **summary.Revenue** sheet: **3-year average sales** (a size measure)

That workbook uses short uppercase names (`CVTR`, `JIM HAWK`), so `TIER_NAME_MAP` maps them to Airtable names.
⚠️ Entries marked `# ?` are guesses or unresolved and need confirming (see the flags in section 2).

In [ ]:
TIER_FILE = '2026_RMF_Budget_Dealer_Tiers.xlsx'

TIER_NAME_MAP = {
    'ATLANTIC GREAT DANE': 'Atlantic Great Dane',
    'JIM HAWK': 'Jim Hawk Truck Trailers',
    'CVTR': 'Central Valley Trailer Repair',
    'CROSSROADS': 'Crossroads Trailer Sales & Service',
    'TOTAL TRAILERS': 'Total Trailers',
    'WALLWORK TRUCK CENTER': 'Wallwork Truck Center',
    'WESTERN': 'Western Trailer Sales Company',
    'GREAT DANE OF UTAH': 'Great Dane of Utah',
    'MOUNT TRAILER': 'Mount Trailer',
    'GLASVAN': 'Glasvan Great Dane Sales',
    'NORTRACK': 'Nortrack De Mexico SA de CV/ Zapata',      # ? tier sheet also lists 'NORTRACK / ZAPATA' separately
    'MAXIM': 'Maxim Truck & Trailers',
    'MACKAY': "MacKay's Truck & Trailer Center",
    'AMERICAN EQUIPMENT': 'American Equipment & Trailers',
    'H.A. DEHART': 'H. A. DeHart & Son',
    'GREAT DANE MONTREAL': 'Great Dane of Montreal',
    'GREAT DANE COLOMBIA': 'Great Dane Colombia',
    'LONESTAR': 'Lonestar',
    'LUBBOCK': 'Lubbock Truck Sales',
    'SOUTHWEST GREAT DANE': 'Southwest Great Dane',
    'COMMERCIAL KAUFMANN': 'Comercial Kaufmann',
    'COX AUTOMOTIVE': 'Cox Automotive',
    # ? Unresolved, no obvious Airtable match: TRUDELL, NORTHEAST GREAT DANE, TE EL PASO, NORTHWEST
    # ? Airtable dealer with no tier-sheet row: Doggett
}

final = pd.read_excel(TIER_FILE, sheet_name='FINAL', header=None)
tiers = final.iloc[5:33, [0, 5, 6]].copy()
tiers.columns = ['Tier Sheet Name', 'Dealer Score', 'Tier']
tiers = tiers.dropna(subset=['Tier Sheet Name'])
tiers['Tier Sheet Name'] = tiers['Tier Sheet Name'].astype(str).str.strip()

size = pd.read_excel(TIER_FILE, sheet_name='summary.Revenue', header=3).iloc[:, [0, 7]]
size.columns = ['Tier Sheet Name', 'Sales 3yr Avg']
size = size.dropna(subset=['Tier Sheet Name'])
size['Tier Sheet Name'] = size['Tier Sheet Name'].astype(str).str.strip()
size['Sales 3yr Avg'] = pd.to_numeric(size['Sales 3yr Avg'], errors='coerce')

tier_ref = tiers.merge(size, on='Tier Sheet Name', how='outer')
tier_ref[key] = tier_ref['Tier Sheet Name'].map(TIER_NAME_MAP)
tier_ref['Tier'] = pd.to_numeric(tier_ref['Tier'], errors='coerce')
tier_ref['Dealer Score'] = pd.to_numeric(tier_ref['Dealer Score'], errors='coerce')

# Allocation size is set by tier, so it's a fallback tier for dealers the tier sheet doesn't cover
ALLOCATION_TIER = {5750: 1, 3750: 2, 2875: 3, 1500: 4}

dealer_ref = dealers[[key, 'Channel Type', 'RSM', 'Marketing Fund Budget']].merge(
    tier_ref.dropna(subset=[key])[[key, 'Tier', 'Dealer Score', 'Sales 3yr Avg']], on=key, how='left', validate='1:1')
dealer_ref['Tier (allocation)'] = dealer_ref['Marketing Fund Budget'].map(ALLOCATION_TIER)
dealer_ref['Tier (used)'] = dealer_ref['Tier'].fillna(dealer_ref['Tier (allocation)'])
dealer_ref['Size band'] = pd.qcut(dealer_ref['Sales 3yr Avg'], 3, labels=['Small', 'Mid', 'Large'])
dealer_ref[dealer_ref['Marketing Fund Budget'] > 0]

## 2. Data-quality checks

Every check adds a row to `dq` (check, frame, severity, rows affected) and keeps the offending rows in `dq_rows[check]`.
Severity: **error** = numbers won't reconcile; **warn** = likely bad or test data; **info** = worth knowing.

In [ ]:
dq, dq_rows = [], {}
TOL = 1.0  # $ tolerance for reconciliation checks

def check(name, frame, severity, rows, note=''):
    rows = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    dq.append({'check': name, 'frame': frame, 'severity': severity, 'rows': len(rows), 'note': note})
    if len(rows):
        dq_rows[name] = rows

# --- Keys & joins -------------------------------------------------------------
for nm, df in [('dealers', dealers), ('budgets', budgets), ('teams', teams)]:
    check(f'{nm}: duplicate dealer names', nm, 'error', df[df[key].duplicated(keep=False)][[key]])
    check(f'{nm}: missing dealer name', nm, 'error', df[df[key].isna()])

all_names = set(dealers[key])
for nm, df in [('budgets', budgets), ('teams', teams)]:
    check(f'{nm}: dealer not in Dealers grid', nm, 'error', df[~df[key].isin(all_names)][[key]])
    check(f'{nm}: Dealers-grid dealer missing here', nm, 'warn',
          dealers[~dealers[key].isin(set(df[key]))][[key]])

opp_dealers = opps.explode('Dealer List')
check('opps: Dealer/Channel not in Dealers grid', 'opps', 'error',
      opp_dealers[~opp_dealers['Dealer List'].isin(all_names)][['Opportunity Name / ID', 'Dealer/Channel']])
check('opps: linked to multiple dealers', 'opps', 'warn',
      opps[opps['Dealer List'].str.len() > 1][['Opportunity Name / ID', 'Dealer/Channel', 'Opportunity Amount']],
      'amount would double-count if summed per dealer')

# --- Opportunities --------------------------------------------------------------
check('opps: duplicate Opportunity URL', 'opps', 'error',
      opps[opps['Opportunity URL'].duplicated(keep=False)])
check('opps: duplicate opportunity name (different URL)', 'opps', 'info',
      opps[opps['Opportunity Name / ID'].duplicated(keep=False)
           & ~opps['Opportunity URL'].duplicated(keep=False)]
          [['Opportunity Name / ID', 'Opportunity URL', 'Opportunity Amount', 'Created Date']])
check('opps: missing amount', 'opps', 'warn',
      opps[opps['Opportunity Amount'].isna()][['Opportunity Name / ID', 'Opportunity Stage']])
check('opps: zero/negative amount', 'opps', 'warn',
      opps[opps['Opportunity Amount'] <= 0][['Opportunity Name / ID', 'Opportunity Amount']])
check('opps: close date before created date', 'opps', 'warn',
      opps[opps['Close Date'] < opps['Created Date']][['Opportunity Name / ID', 'Created Date', 'Close Date', 'Opportunity Stage']])
check('opps: open but close date already passed', 'opps', 'info',
      opps[~opps['Is Closed'] & (opps['Close Date'] < AS_OF)][['Opportunity Name / ID', 'Opportunity Stage', 'Close Date']],
      f'stale pipeline as of {AS_OF.date()}')
check('opps: missing RSM', 'opps', 'warn',
      opps[opps['RSM (from Dealer/Channel)'].isna()][['Opportunity Name / ID', 'Dealer/Channel']])
check('opps: pitched-and-opened but not pitched', 'opps', 'error',
      opps[opps['Pitched & Opened?'] & ~opps['Pitched? (any)']][['Opportunity Name / ID']])
check('opps: opened pitches > pitches', 'opps', 'error',
      opps[opps['# Opened Pitches to Acct'] > opps['# Pitches to Acct']][['Opportunity Name / ID', '# Pitches to Acct', '# Opened Pitches to Acct']])

# Same email domain, different spelling of the account name (e.g. Berger vs Burger)
dom = opps.dropna(subset=['Account Domain/Email'])
names_per_dom = dom.groupby('Account Domain/Email')['Customer Accounts Touched'].unique()
check('opps: one domain, several account names', 'opps', 'warn',
      names_per_dom[names_per_dom.str.len() > 1].rename('account names').reset_index())

# --- Dealers ----------------------------------------------------------------------
check('dealers: missing RSM', 'dealers', 'warn', dealers[dealers['RSM'].isna()][[key]])
check('dealers: missing Join Date', 'dealers', 'info', dealers[dealers['Join Date'].isna()][[key, 'Channel Type']])

# --- Budget & spend reconciliation (Dealers grid vs Dealer & Budgets grid) ---------
recon = dealers[[key, 'Marketing Fund Budget', 'Marketing Fund Spend', 'Growth Fund Spend', 'Signage Spend',
                 'Marketing Fund Spend %', 'Promo Wallet Spend', 'Promo Wallet Usage %',
                 'Total Reimbursement Program Spend', 'Total Marketing Support Spend',
                 'Total Marketing Support Spend (excluding Signage)']].merge(
    budgets[[key, 'Marketing Fund Budget', 'Marketing Fund Approved Total', 'Growth Fund Approved Total',
             'Signage Approved Total', 'Marketing Remaining']],
    on=key, how='outer', suffixes=('', ' (budgets)'))
recon = recon.fillna({c: 0 for c in recon.select_dtypes('number').columns if 'Usage' not in c})

def mismatch(a, b):
    return recon[(recon[a] - recon[b]).abs() > TOL][[key, a, b]]

check('MF budget differs between grids', 'recon', 'error', mismatch('Marketing Fund Budget', 'Marketing Fund Budget (budgets)'))
check('MF spend != MF approved total', 'recon', 'error', mismatch('Marketing Fund Spend', 'Marketing Fund Approved Total'))
check('Growth spend != Growth approved total', 'recon', 'error', mismatch('Growth Fund Spend', 'Growth Fund Approved Total'))
check('Signage spend != Signage approved total', 'recon', 'error', mismatch('Signage Spend', 'Signage Approved Total'))

recon['remaining_calc'] = recon['Marketing Fund Budget'] - recon['Marketing Fund Spend']
check('Marketing Remaining != budget - spend', 'recon', 'error', mismatch('Marketing Remaining', 'remaining_calc'))

recon['reimb_calc'] = recon[['Marketing Fund Spend', 'Growth Fund Spend', 'Signage Spend']].sum(axis=1)
check('Total Reimbursement Spend != MF + Growth + Signage', 'recon', 'error',
      mismatch('Total Reimbursement Program Spend', 'reimb_calc'))
recon['support_calc'] = recon['reimb_calc'] + recon['Promo Wallet Spend']
check('Total Marketing Support Spend != reimbursement + promo', 'recon', 'error',
      mismatch('Total Marketing Support Spend', 'support_calc'))
recon['support_ex_sign_calc'] = recon['support_calc'] - recon['Signage Spend']
check('Support Spend (ex Signage) != total - signage', 'recon', 'error',
      mismatch('Total Marketing Support Spend (excluding Signage)', 'support_ex_sign_calc'))

recon['mf_pct_calc'] = (recon['Marketing Fund Spend'] / recon['Marketing Fund Budget']).where(recon['Marketing Fund Budget'] > 0, 0)
check('MF Spend % != spend / budget', 'recon', 'warn',
      recon[(recon['Marketing Fund Spend %'] - recon['mf_pct_calc']).abs() > 0.01][[key, 'Marketing Fund Spend %', 'mf_pct_calc', 'Marketing Fund Spend', 'Marketing Fund Budget']])

# Promo Wallet allocation isn't exported. If it equals the MF budget, spend / MF budget
# rounded to a whole % should reproduce the exported Promo Wallet Usage %.
recon['pw_usage_calc'] = (recon['Promo Wallet Spend'] / recon['Marketing Fund Budget']).where(recon['Marketing Fund Budget'] > 0)
check('Promo Wallet allocation != MF budget', 'recon', 'warn',
      recon[recon['Promo Wallet Usage %'].notna()
            & ((recon['pw_usage_calc'].round(2) - recon['Promo Wallet Usage %']).abs() > 0.005)]
          [[key, 'Promo Wallet Spend', 'Promo Wallet Usage %', 'pw_usage_calc', 'Marketing Fund Budget']],
      'Usage % is exported as a whole number, so compare at 2 decimals')

check('MF spend with no MF budget', 'recon', 'warn',
      recon[(recon['Marketing Fund Budget'] == 0) & (recon['Marketing Fund Spend'] > 0)][[key, 'Marketing Fund Spend']])
check('Promo spend with no allocation', 'recon', 'warn',
      recon[(recon['Marketing Fund Budget'] == 0) & (recon['Promo Wallet Spend'] > 0)][[key, 'Promo Wallet Spend']])
check('MF spend over budget', 'recon', 'warn',
      recon[(recon['Marketing Fund Budget'] > 0) & (recon['Marketing Fund Spend'] > recon['Marketing Fund Budget'] + TOL)])
check('Promo spend over allocation', 'recon', 'warn',
      recon[(recon['Marketing Fund Budget'] > 0) & (recon['Promo Wallet Spend'] > recon['Marketing Fund Budget'] + TOL)])
check('negative Marketing Remaining', 'recon', 'warn', recon[recon['Marketing Remaining'] < 0][[key, 'Marketing Remaining']])

# --- Reference-data crosswalk -------------------------------------------------
check('tier sheet: dealer not mapped to Airtable', 'tier_ref', 'warn',
      tier_ref[tier_ref[key].isna() & tier_ref['Tier'].notna()][['Tier Sheet Name', 'Tier', 'Sales 3yr Avg']],
      'add to TIER_NAME_MAP if it is an Airtable dealer')
check('allocated dealer with no tier-sheet tier', 'tier_ref', 'warn',
      dealer_ref[(dealer_ref['Marketing Fund Budget'] > 0) & dealer_ref['Tier'].isna()][[key, 'Marketing Fund Budget', 'Tier (allocation)']])
check('tier sheet tier != tier implied by allocation', 'tier_ref', 'warn',
      dealer_ref[dealer_ref['Tier'].notna() & dealer_ref['Tier (allocation)'].notna()
                 & (dealer_ref['Tier'] != dealer_ref['Tier (allocation)'])][[key, 'Tier', 'Tier (allocation)', 'Marketing Fund Budget']])
check('allocation not a standard tier amount', 'tier_ref', 'warn',
      dealer_ref[(dealer_ref['Marketing Fund Budget'] > 0) & dealer_ref['Tier (allocation)'].isna()][[key, 'Marketing Fund Budget']])

# --- Revenue / pipeline touched reconciliation ------------------------------------
rt = budgets[[key, 'Marketing Fund Revenue Touched', 'Growth Fund Revenue Touched', 'Total Reimbursement Revenue Touched',
              'Marketing Fund Pipeline Touched', 'Growth Fund Pipeline Touched', 'Total Reimbursement Pipeline Touched']].merge(
    dealers[[key, 'Reimbursement Revenue Touched', 'Reimbursement Pipeline Touched', 'Highspot Revenue Touched',
             'Promo Revenue Touched', 'Total Revenue Touched', 'Total Marketing Support Spend', 'Revenue Touched Multiple']],
    on=key, how='outer').fillna(0)

def rt_mismatch(a, b):
    return rt[(rt[a] - rt[b]).abs() > TOL][[key, a, b]]

rt['mf_gf_rev'] = rt['Marketing Fund Revenue Touched'] + rt['Growth Fund Revenue Touched']
check('Reimb. revenue touched != MF + Growth revenue touched', 'revenue', 'error', rt_mismatch('Total Reimbursement Revenue Touched', 'mf_gf_rev'))
rt['mf_gf_pipe'] = rt['Marketing Fund Pipeline Touched'] + rt['Growth Fund Pipeline Touched']
check('Reimb. pipeline touched != MF + Growth pipeline touched', 'revenue', 'error', rt_mismatch('Total Reimbursement Pipeline Touched', 'mf_gf_pipe'))
check('Reimb. revenue touched differs between grids', 'revenue', 'error', rt_mismatch('Reimbursement Revenue Touched', 'Total Reimbursement Revenue Touched'))
rt['total_rev_calc'] = rt[['Highspot Revenue Touched', 'Promo Revenue Touched', 'Reimbursement Revenue Touched']].sum(axis=1)
check('Total Revenue Touched != Highspot + Promo + Reimbursement', 'revenue', 'error', rt_mismatch('Total Revenue Touched', 'total_rev_calc'))
check('revenue touched includes Highspot $ (no support spend)', 'revenue', 'info',
      rt[rt['Highspot Revenue Touched'] > 0][[key, 'Highspot Revenue Touched', 'Total Revenue Touched', 'Total Marketing Support Spend']],
      'Revenue Touched Multiple puts Highspot revenue over support spend that excludes Highspot')
check('revenue touched with no support spend', 'revenue', 'warn',
      rt[(rt['Total Revenue Touched'] > 0) & (rt['Total Marketing Support Spend'] <= 0)][[key, 'Total Revenue Touched']])

# --- Test / placeholder records ------------------------------------------------------
TEST_PAT = r'\b(?:test\w*|tets|unnamed record|dummy|sample)\b'
test_hits = []
for nm, df in [('dealers', dealers), ('budgets', budgets), ('teams', teams), ('opps', opps)]:
    for c in text_cols(df):
        if c == 'Dealer List':
            continue
        m = df[c].astype('string').str.contains(TEST_PAT, case=False, regex=True).fillna(False)
        for i in df.index[m]:
            test_hits.append({'frame': nm, 'row': i, 'dealer/opp': df.iloc[:, 0][i], 'column': c, 'value': df.at[i, c]})
check('test / placeholder records', 'all', 'warn', pd.DataFrame(test_hits),
      'linked request names only; rollup $ may include them')

dq = pd.DataFrame(dq)
sev_order = {'error': 0, 'warn': 1, 'info': 2}
dq['_sev'] = dq['severity'].map(sev_order)
dq = dq.sort_values(['_sev', 'rows'], ascending=[True, False]).drop(columns='_sev').reset_index(drop=True)
dq.query('rows > 0')

In [ ]:
# Checks that passed (0 rows), for the record
dq.query('rows == 0')[['check', 'frame']]

In [ ]:
# Drill into any failing check: inspect every flagged row
for name, rows in dq_rows.items():
    print(f'\n=== {name} ({len(rows)}) ===')
    display(rows)

## 3. Budget utilization by fund type

**Question:** What share of allocated budget has been used, by fund type?

- **Marketing Fund**: allocation = `Marketing Fund Budget`; used = `Marketing Fund Spend`
- **Promo Wallet**: allocation isn't exported. Section 2 checks that the allocation implied by `Promo Wallet Usage %` equals the Marketing Fund budget, and this section uses the MF budget as the allocation.
- **Growth Fund / Signage**: no allocation in any export, so they're shown as spend only

Spend recorded against dealers with no allocation is reported separately rather than silently inflating the % used.

In [ ]:
b = recon.copy()
has_budget = b['Marketing Fund Budget'] > 0

rows = []
for fund, alloc_col, spend_col in [('Marketing Fund', 'Marketing Fund Budget', 'Marketing Fund Spend'),
                                   ('Promo Wallet',   'Marketing Fund Budget', 'Promo Wallet Spend'),
                                   ('Growth Fund',    None,                    'Growth Fund Spend'),
                                   ('Signage',        None,                    'Signage Spend')]:
    allocated = b.loc[has_budget, alloc_col].sum() if alloc_col else np.nan
    used_in   = b.loc[has_budget, spend_col].sum() if alloc_col else np.nan
    rows.append({
        'Fund': fund,
        'Allocated': allocated,
        'Used (against allocation)': used_in,
        'Used (no allocation)': b.loc[~has_budget, spend_col].sum() if alloc_col else b[spend_col].sum(),
        'Dealers w/ allocation': has_budget.sum() if alloc_col else np.nan,
        'Dealers spending': (b[spend_col] > 0).sum(),
    })

utilization = pd.DataFrame(rows).set_index('Fund')
utilization['Remaining'] = utilization['Allocated'] - utilization['Used (against allocation)']
utilization['% Used'] = utilization['Used (against allocation)'] / utilization['Allocated']

utilization.style.format({
    'Allocated': '${:,.0f}', 'Used (against allocation)': '${:,.0f}', 'Used (no allocation)': '${:,.0f}',
    'Remaining': '${:,.0f}', '% Used': '{:.1%}', 'Dealers w/ allocation': '{:.0f}',
}, na_rep='—')

In [ ]:
# Per-dealer utilization, for dealers with an allocation
per_dealer = b.loc[has_budget, [key, 'Marketing Fund Budget', 'Marketing Fund Spend', 'Promo Wallet Spend']].copy()
per_dealer['MF % Used'] = per_dealer['Marketing Fund Spend'] / per_dealer['Marketing Fund Budget']
per_dealer['Promo % Used'] = per_dealer['Promo Wallet Spend'] / per_dealer['Marketing Fund Budget']
per_dealer.sort_values('Promo % Used', ascending=False).style.format(
    {'Marketing Fund Budget': '${:,.0f}', 'Marketing Fund Spend': '${:,.0f}', 'Promo Wallet Spend': '${:,.0f}',
     'MF % Used': '{:.0%}', 'Promo % Used': '{:.0%}'})

In [ ]:
plot = utilization.dropna(subset=['% Used'])['% Used']
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.barh(plot.index, plot.values, color='#4C72B0')
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('Share of allocation used')
ax.set_title('Budget utilization by fund type')
ax.invert_yaxis()
for i, v in enumerate(plot.values):
    ax.text(v + 0.01, i, f'{v:.1%}', va='center')
plt.tight_layout()

## 4. Dealer adoption by fund type & program component

**Question:** What share of dealers have used each fund type or program component at all?

A dealer counts as having used a component if:

| Component | Used if |
|---|---|
| Marketing Fund / Growth Fund / Signage / Promo Wallet | spend > $0 |
| Pre-Approvals | at least one non-test pre-approval linked |
| Highspot | `Highspot Pitches Sent` > 0 |
| Dealer Portal | `total Portal Logins YTD` > 0 |
| Engagement events | at least one non-test attendance record |
| Campaign & Creative | at least one non-test activity linked |
| BuyGreatDane.com page | page marked active |

Linked records whose name looks like test data (`test`, `tets`, …, see section 2) are **excluded** by default; set `EXCLUDE_TEST_RECORDS = False` to count them.
`Unnamed record` is treated as real because Airtable uses it for records without a name, but it's still flagged in section 2.

Shares are shown for all dealers and by channel type. `National Accounts` isn't a dealer, so it's excluded unless `INCLUDE_NATIONAL_ACCOUNTS = True`.

In [ ]:
EXCLUDE_TEST_RECORDS = True
INCLUDE_NATIONAL_ACCOUNTS = False
TEST_ONLY_PAT = r'\b(?:test\w*|tets|dummy|sample)\b'

def real_link_count(s):
    """Number of linked records in a comma-separated field, optionally ignoring test records."""
    links = split_list(s)
    if EXCLUDE_TEST_RECORDS:
        links = links.apply(lambda xs: [x for x in xs if not re.search(TEST_ONLY_PAT, x, flags=re.I)])
    return links.str.len()

base = dealers[[key, 'Channel Type', 'Marketing Fund Spend', 'Growth Fund Spend', 'Signage Spend',
                'Promo Wallet Spend', 'Highspot Pitches Sent']].merge(
    teams[[key, 'total Portal Logins YTD', 'Engagement Attendance', 'Campaign & Creative Activity',
           'BuyGreatDane.com Page Active? (from BuyGreatDane.com Participation)']], on=key, how='left', validate='1:1').merge(
    budgets[[key, 'Pre-Approvals']], on=key, how='left', validate='1:1')

if not INCLUDE_NATIONAL_ACCOUNTS:
    base = base[base['Channel Type'] != 'National Accounts']

FUNDS = ['Marketing Fund', 'Growth Fund', 'Signage', 'Promo Wallet']
PROGRAMS = ['Pre-Approvals', 'Highspot', 'Dealer Portal', 'Engagement events', 'Campaign & Creative', 'BuyGreatDane.com page']

used = pd.DataFrame({
    'Marketing Fund':        base['Marketing Fund Spend'].fillna(0) > 0,
    'Growth Fund':           base['Growth Fund Spend'].fillna(0) > 0,
    'Signage':               base['Signage Spend'].fillna(0) > 0,
    'Promo Wallet':          base['Promo Wallet Spend'].fillna(0) > 0,
    'Pre-Approvals':         real_link_count(base['Pre-Approvals']) > 0,
    'Highspot':              base['Highspot Pitches Sent'].fillna(0) > 0,
    'Dealer Portal':         base['total Portal Logins YTD'].fillna(0) > 0,
    'Engagement events':     real_link_count(base['Engagement Attendance']) > 0,
    'Campaign & Creative':   real_link_count(base['Campaign & Creative Activity']) > 0,
    'BuyGreatDane.com page': base['BuyGreatDane.com Page Active? (from BuyGreatDane.com Participation)']
                                 .astype('string').str.match(r'^[1-9]'),
}, index=base.index).fillna(False).astype(bool)
used['Any fund'] = used[FUNDS].any(axis=1)
used['Any program component'] = used[PROGRAMS].any(axis=1)
used['Anything at all'] = used.any(axis=1)
used.insert(0, key, base[key])
used.insert(1, 'Channel Type', base['Channel Type'])

print(f'{len(used)} dealers in scope; test records excluded: {EXCLUDE_TEST_RECORDS}')

In [ ]:
components = FUNDS + ['Any fund'] + PROGRAMS + ['Any program component', 'Anything at all']

adoption = pd.DataFrame({'All dealers': used[components].mean()})
for ch, grp in used.groupby('Channel Type'):
    adoption[ch] = grp[components].mean()
adoption.insert(0, 'Dealers using', used[components].sum())

counts = used['Channel Type'].value_counts()
adoption.columns = [c if c in ('Dealers using', 'All dealers') else f'{c} (n={counts[c]})' for c in adoption.columns]
adoption = adoption.rename(columns={'All dealers': f'All dealers (n={len(used)})'})

adoption.style.format({c: '{:.0%}' for c in adoption.columns if c != 'Dealers using'}) \
    .set_properties(subset=pd.IndexSlice[['Any fund', 'Any program component', 'Anything at all'], :],
                    **{'font-weight': 'bold'})

In [ ]:
# Note: corporate-owned branches have no fund allocations, so their fund adoption
# is structurally ~0%. Restrict to dealers with an allocation for a like-for-like view.
eligible = used[used[key].isin(recon.loc[recon['Marketing Fund Budget'] > 0, key])]
print(f'Dealers with a Marketing Fund / Promo Wallet allocation: {len(eligible)}')
eligible[FUNDS + ['Any fund']].mean().to_frame('Share of allocated dealers using').style.format('{:.0%}')

In [ ]:
# Dealer x component grid: who has and hasn't used what
grid = used.set_index(key)[FUNDS + PROGRAMS].apply(lambda col: col.map({True: '●', False: ''}))
grid.insert(0, 'Channel Type', used.set_index(key)['Channel Type'])
grid.insert(1, '# used', used.set_index(key)[FUNDS + PROGRAMS].sum(axis=1))
grid.sort_values(['Channel Type', '# used'], ascending=[True, False])

In [ ]:
plot = used[FUNDS + PROGRAMS].mean().sort_values()
colors = ['#4C72B0' if c in FUNDS else '#DD8452' for c in plot.index]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(plot.index, plot.values, color=colors)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel(f'Share of dealers (n={len(used)})')
ax.set_title('Dealers that have used each fund / program component')
for i, v in enumerate(plot.values):
    ax.text(v + 0.01, i, f'{v:.0%}', va='center')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#4C72B0', label='Fund'), Patch(color='#DD8452', label='Program component')],
          loc='lower right', frameon=False)
plt.tight_layout()

## 5. Allocation used, among dealers who used each fund

**Question:** Among dealers who have used funds, how much of their allocation did they use?

- Only **Marketing Fund** and **Promo Wallet** have allocations (see section 3), so only these two can be measured. Promo Wallet allocation = MF budget.
- For each fund, the population is dealers with an allocation **and** spend > $0 in that fund. Dealers with spend but no allocation (section 2 flags them) are listed separately, because % used is undefined for them.
- **Combined** = (MF spend + Promo spend) / (MF allocation + Promo allocation), for dealers who used either fund.
- Two views of the same thing: **dealer average** (each dealer counts once) and **pooled** (total spend ÷ total allocation, so big allocations weigh more).

In [ ]:
FUNDED = {'Marketing Fund': 'Marketing Fund Spend', 'Promo Wallet': 'Promo Wallet Spend'}
BUCKETS = [0, 0.25, 0.50, 0.75, 1.0 - 1e-9, np.inf]
BUCKET_LABELS = ['<25%', '25–50%', '50–75%', '75–<100%', '100%+']

alloc = recon[[key, 'Marketing Fund Budget'] + list(FUNDED.values())].copy()
alloc = alloc.rename(columns={'Marketing Fund Budget': 'Allocation'})
alloc['Combined Spend'] = alloc[list(FUNDED.values())].sum(axis=1)

# Long format: one row per dealer x fund that was actually used
long = []
for fund, col in {**FUNDED, 'Combined': 'Combined Spend'}.items():
    a = alloc['Allocation'] * (len(FUNDED) if fund == 'Combined' else 1)
    part = pd.DataFrame({key: alloc[key], 'Fund': fund, 'Allocation': a, 'Spend': alloc[col]})
    long.append(part[part['Spend'] > 0])
long = pd.concat(long, ignore_index=True)

no_alloc = long[long['Allocation'] <= 0]
usage = long[long['Allocation'] > 0].copy()
usage['% Used'] = usage['Spend'] / usage['Allocation']
usage['Bucket'] = pd.cut(usage['% Used'], BUCKETS, labels=BUCKET_LABELS, right=False)

print('Spend with no allocation (excluded from % used):')
display(no_alloc)

In [ ]:
def summarize(g):
    return pd.Series({
        'Dealers who used it':       len(g),
        'Allocation (those dealers)': g['Allocation'].sum(),
        'Spend':                     g['Spend'].sum(),
        'Pooled % used':             g['Spend'].sum() / g['Allocation'].sum(),
        'Dealer avg % used':         g['% Used'].mean(),
        'Median % used':             g['% Used'].median(),
        'Min % used':                g['% Used'].min(),
        'Max % used':                g['% Used'].max(),
        'Fully used (≥100%)':        (g['% Used'] >= 1 - 1e-9).sum(),
        'Unused allocation':         (g['Allocation'] - g['Spend']).clip(lower=0).sum(),
    })

fund_order = list(FUNDED) + ['Combined']
usage_summary = usage.groupby('Fund').apply(summarize, include_groups=False).reindex(fund_order)
usage_summary.style.format({
    'Dealers who used it': '{:.0f}', 'Fully used (≥100%)': '{:.0f}',
    'Allocation (those dealers)': '${:,.0f}', 'Spend': '${:,.0f}', 'Unused allocation': '${:,.0f}',
    'Pooled % used': '{:.0%}', 'Dealer avg % used': '{:.0%}', 'Median % used': '{:.0%}',
    'Min % used': '{:.0%}', 'Max % used': '{:.0%}',
}, na_rep='—')

In [ ]:
# How used dealers are distributed across utilization bands
pd.crosstab(usage['Fund'], usage['Bucket'], dropna=False).reindex(fund_order)

In [ ]:
# Per dealer: % of each allocation used, for dealers who used at least one fund
per_dealer_use = usage.pivot_table(index=key, columns='Fund', values='% Used').reindex(columns=fund_order)
per_dealer_use.insert(0, 'Allocation', alloc.set_index(key)['Allocation'])
per_dealer_use.sort_values('Combined', ascending=False).style.format(
    {'Allocation': '${:,.0f}', **{f: '{:.0%}' for f in fund_order}}, na_rep='not used')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for i, fund in enumerate(fund_order):
    vals = usage.loc[usage['Fund'] == fund, '% Used']
    jitter = np.random.default_rng(0).uniform(-0.12, 0.12, len(vals))
    ax.scatter(vals, np.full(len(vals), i) + jitter, alpha=0.7, s=40, color='#4C72B0')
    ax.plot([vals.median()] * 2, [i - 0.3, i + 0.3], color='#C44E52', lw=2)
ax.axvline(1, color='grey', ls=':', lw=1)
ax.set_yticks(range(len(fund_order)), fund_order)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlim(0, max(1.05, usage['% Used'].max() + 0.05))
ax.set_xlabel('Share of allocation used (one dot per dealer; red line = median)')
ax.set_title('Allocation used among dealers who used each fund')
plt.tight_layout()

# How funds get used

## 6. What are funds most commonly used for?

⚠️ **Data gap:** the exports have no activity-type field. Each request only carries a free-text name (e.g. *Doggett Golf Tournament Gold Sponsorship*).
Requests are classified by keyword using `ACTIVITY_RULES` below. Edit the rules, or pin an individual request with `ACTIVITY_OVERRIDES`. Anything unmatched shows as **Unclassified** so it can't silently drop out.

Sources: Marketing Fund, Growth Fund and Signage requests (Dealer & Budgets grid) and Promo Wallet orders (Dealer Teams grid). Pre-approvals and test records are excluded.

In [ ]:
# First match wins, so put more specific rules first
ACTIVITY_RULES = [
    ('Trade association / convention', r'association|convention|conference|expo|canacar|show'),
    ('Golf / tournament sponsorship',   r'golf|tournament'),
    ('Charity / community event',       r'charity|bbq|foodbank|fundraiser|community'),
    ('Sports team sponsorship',         r'wildcats|hockey|football|baseball|team'),
    ('Signage',                         r'signage|\\bsign\\b'),
    ('Customer gifts',                  r'gift|handout|christmas'),
    ('Promo products',                  r'promo|pormo|order|stock'),
]
FUND_DEFAULT_ACTIVITY = {'Promo Wallet': 'Promo products', 'Signage': 'Signage'}   # used when no rule matches
ACTIVITY_OVERRIDES = {}   # e.g. {'Midwest Gordon Foods WIT Promo': 'Trade association / convention'}

REQUEST_SOURCES = [   # (fund, frame, request-name column, $ column, revenue-touched column, pipeline-touched column)
    ('Marketing Fund', budgets, 'Marketing Fund Requests', 'Marketing Fund Approved Total', 'Marketing Fund Revenue Touched', 'Marketing Fund Pipeline Touched'),
    ('Growth Fund',    budgets, 'Growth Fund Requests',    'Growth Fund Approved Total',    'Growth Fund Revenue Touched',    'Growth Fund Pipeline Touched'),
    ('Signage',        budgets, 'Signage Requests',        'Signage Approved Total',        None, None),
    ('Promo Wallet',   teams,   'Promo Wallet Orders',     None,                            None, None),
]

def classify(name, fund):
    if name in ACTIVITY_OVERRIDES:
        return ACTIVITY_OVERRIDES[name]
    for label, pat in ACTIVITY_RULES:
        if re.search(pat, name, flags=re.I):
            return label
    return FUND_DEFAULT_ACTIVITY.get(fund, 'Unclassified')

promo_money = dealers.set_index(key)[['Promo Wallet Spend', 'Promo Revenue Touched', 'Promo Pipeline Touched']]

req_rows = []
for fund, frame, name_col, amt_col, rev_col, pipe_col in REQUEST_SOURCES:
    f = frame[[key, name_col]].copy()
    f['Request'] = split_list(f[name_col])
    f = f.explode('Request').dropna(subset=['Request'])
    f = f[~f['Request'].str.contains(TEST_ONLY_PAT, case=False, regex=True)]
    f['Fund'] = fund
    f['Requests for dealer in fund'] = f.groupby(key)['Request'].transform('size')
    if fund == 'Promo Wallet':
        totals = promo_money.reindex(f[key]).to_numpy()
        f['Dealer fund $'], f['Dealer fund revenue touched'], f['Dealer fund pipeline touched'] = totals.T
    else:
        lookup = frame.set_index(key)
        f['Dealer fund $'] = lookup[amt_col].reindex(f[key]).to_numpy()
        f['Dealer fund revenue touched'] = lookup[rev_col].reindex(f[key]).to_numpy() if rev_col else np.nan
        f['Dealer fund pipeline touched'] = lookup[pipe_col].reindex(f[key]).to_numpy() if pipe_col else np.nan
    req_rows.append(f.drop(columns=name_col))

requests = pd.concat(req_rows, ignore_index=True)
requests['Activity Type'] = [classify(n, fd) for n, fd in zip(requests['Request'], requests['Fund'])]
requests[[key, 'Fund', 'Request', 'Activity Type', 'Requests for dealer in fund']]

In [ ]:
# Most common uses: count of requests by activity type (overall and by fund)
activity_counts = pd.crosstab(requests['Activity Type'], requests['Fund'], margins=True, margins_name='Total')
activity_counts.drop(index='Total').sort_values('Total', ascending=False).pipe(
    lambda t: pd.concat([t, activity_counts.loc[['Total']]]))

In [ ]:
# Unclassified requests: add a rule or override for these
requests.loc[requests['Activity Type'] == 'Unclassified', [key, 'Fund', 'Request']]

## 7. Average reimbursement per activity type

⚠️ **Data gap:** $ amounts are only exported as **dealer-level totals per fund**, not per request.
- A request gets an exact $ amount only when it's the **only** request that dealer has in that fund.
- When a dealer has several requests in a fund, the amount is unknown. They're excluded by default; set `SPLIT_EVENLY = True` to divide the dealer total evenly across them (an estimate).

The coverage columns show how many requests have a known amount.

In [ ]:
SPLIT_EVENLY = False

requests['Amount'] = np.where(requests['Requests for dealer in fund'] == 1, requests['Dealer fund $'],
                              requests['Dealer fund $'] / requests['Requests for dealer in fund'] if SPLIT_EVENLY else np.nan)
requests['Amount is estimate'] = SPLIT_EVENLY & (requests['Requests for dealer in fund'] > 1)

# A request whose dealer shows $0 spend in that fund has no recorded amount; don't count it as a $0 activity
zero_spend = requests['Dealer fund $'].fillna(0) <= 0
requests.loc[zero_spend, 'Amount'] = np.nan
if zero_spend.any():
    print('⚠️ Requests linked but no $ recorded for the dealer in that fund (excluded from averages):')
    display(requests.loc[zero_spend, [key, 'Fund', 'Request']])

avg_by_activity = requests.groupby('Activity Type').agg(
    Requests=('Request', 'size'),
    Requests_with_amount=('Amount', 'count'),
    Total=('Amount', 'sum'),
    Average=('Amount', 'mean'),
    Median=('Amount', 'median'),
    Min=('Amount', 'min'),
    Max=('Amount', 'max'),
).sort_values('Average', ascending=False)
avg_by_activity['Coverage'] = avg_by_activity['Requests_with_amount'] / avg_by_activity['Requests']
avg_by_activity.style.format({c: '${:,.0f}' for c in ['Total', 'Average', 'Median', 'Min', 'Max']} | {'Coverage': '{:.0%}'}, na_rep='—')

In [ ]:
# Same, split by fund
requests.pivot_table(index='Activity Type', columns='Fund', values='Amount', aggfunc='mean') \
    .style.format('${:,.0f}', na_rep='—')

## 8. Time from request to reimbursement

⛔ **Can't be answered from the current exports.** No request, approval or reimbursement dates are included; the fund request fields are just record names.

**Needed:** a request-level export from the Marketing Fund / Growth Fund / Signage / Promo Wallet request tables with
`Request name, Dealer, Fund, Submitted date, Approved date, Reimbursed/paid date, Amount`.
Once it exists, put the filename in `REQUESTS_FILE` and the cell below will calculate the days and break them down by fund and activity type.

In [ ]:
REQUESTS_FILE = None   # e.g. 'Reimbursement Requests_Grid view_2026-10-07.xlsx'
DATE_COLS = {'submitted': 'Submitted Date', 'reimbursed': 'Reimbursed Date', 'fund': 'Fund', 'name': 'Request Name'}

if REQUESTS_FILE:
    rq = clean_frame(pd.read_excel(REQUESTS_FILE), 'requests', dates=[DATE_COLS['submitted'], DATE_COLS['reimbursed']])
    rq['Days to reimburse'] = (rq[DATE_COLS['reimbursed']] - rq[DATE_COLS['submitted']]).dt.days
    rq['Activity Type'] = [classify(str(n), f) for n, f in zip(rq[DATE_COLS['name']], rq[DATE_COLS['fund']])]
    print('Not yet reimbursed:', rq['Days to reimburse'].isna().sum(), '| negative durations:', (rq['Days to reimburse'] < 0).sum())
    display(rq.groupby(DATE_COLS['fund'])['Days to reimburse'].describe())
    display(rq.groupby('Activity Type')['Days to reimburse'].describe())
else:
    print('REQUESTS_FILE not set; see the note above.')

# Dealer network

A per-dealer frame (`net`) is used for sections 9–12. It combines allocation, spend, % used, tier, size, RSM and program activity.
Population: dealers with an allocation, since % used is undefined without one.

In [ ]:
net = dealer_ref.merge(
    recon[[key, 'Marketing Fund Spend', 'Promo Wallet Spend', 'Growth Fund Spend', 'Signage Spend', 'Total Marketing Support Spend']],
    on=key, how='left', validate='1:1').merge(
    used.drop(columns='Channel Type'), on=key, how='left', validate='1:1').merge(
    dealers[[key, 'Highspot Pitches Sent', 'Portal Adoption %', 'Overall Engagement Score', 'Total Revenue Touched',
             'Total Pipeline Touched', 'Years Since Joined']], on=key, how='left', validate='1:1').merge(
    teams[[key, 'total Portal Logins YTD', 'Total Activities Participated In']], on=key, how='left', validate='1:1')

net = net[net['Marketing Fund Budget'] > 0].copy()
net['Allocation (MF + Promo)'] = net['Marketing Fund Budget'] * 2
net['Allocated spend'] = net['Marketing Fund Spend'] + net['Promo Wallet Spend']
net['% Used (MF + Promo)'] = net['Allocated spend'] / net['Allocation (MF + Promo)']
net['MF % Used'] = net['Marketing Fund Spend'] / net['Marketing Fund Budget']
net['Promo % Used'] = net['Promo Wallet Spend'] / net['Marketing Fund Budget']
net['# Components used'] = net[FUNDS + PROGRAMS].sum(axis=1)
print(len(net), 'dealers with an allocation')

## 9. Usage by dealer tier and size

- **Tier** comes from the tier workbook. Dealers it doesn't cover fall back to the tier implied by allocation size (see `Tier (used)` and the section 2 flags).
- **Size** = 3-year average sales from the tier workbook, split into thirds. Dealers with no sales figure are shown as *Unknown*.

In [ ]:
def usage_by(df, col):
    g = df.groupby(col, dropna=False, observed=True)
    out = pd.DataFrame({
        'Dealers': g.size(),
        'Allocation': g['Allocation (MF + Promo)'].sum(),
        'Spend': g['Allocated spend'].sum(),
        'Pooled % used': g['Allocated spend'].sum() / g['Allocation (MF + Promo)'].sum(),
        'Avg dealer % used': g['% Used (MF + Promo)'].mean(),
        '% using any fund': g['Any fund'].mean(),
        '% using MF': g['Marketing Fund'].mean(),
        '% using Promo': g['Promo Wallet'].mean(),
        'Growth + Signage $': g['Growth Fund Spend'].sum() + g['Signage Spend'].sum(),
        'Avg components used': g['# Components used'].mean(),
    })
    return out

usage_fmt = {'Allocation': '${:,.0f}', 'Spend': '${:,.0f}', 'Growth + Signage $': '${:,.0f}',
             'Pooled % used': '{:.0%}', 'Avg dealer % used': '{:.0%}', '% using any fund': '{:.0%}',
             '% using MF': '{:.0%}', '% using Promo': '{:.0%}', 'Avg components used': '{:.1f}'}

usage_by(net, 'Tier (used)').style.format(usage_fmt)

In [ ]:
net['Size band'] = net['Size band'].cat.add_categories('Unknown').fillna('Unknown')
usage_by(net, 'Size band').style.format(usage_fmt)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
known = net.dropna(subset=['Sales 3yr Avg'])
ax.scatter(known['Sales 3yr Avg'], known['% Used (MF + Promo)'], s=known['Marketing Fund Budget'] / 40, alpha=0.6, color='#4C72B0')
for _, r in known.iterrows():
    ax.annotate(r[key], (r['Sales 3yr Avg'], r['% Used (MF + Promo)']), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_xscale('log')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('3-yr average sales (log scale)')
ax.set_ylabel('% of MF + Promo allocation used')
ax.set_title('Usage vs dealer size (bubble = allocation)')
plt.tight_layout()
print('Spearman rank correlation, size vs % used:',
      round(spearman(known['Sales 3yr Avg'], known['% Used (MF + Promo)']), 2), f'(n={len(known)})')

## 10. Highest and lowest users relative to allocation, and what they have in common

Dealers are ranked by % of their MF + Promo allocation used and grouped as **Non-user** (0% used), **Lower user** and **Higher user**. Users are split at their median.
Equal thirds would put tied 0% dealers into different groups arbitrarily, because many dealers sit at exactly 0%.
The profile table compares each third on tier, size, channel, RSM mix and engagement with other program components.

In [ ]:
pct_used = net['% Used (MF + Promo)']
users_median = pct_used[pct_used > 0].median()
net['Usage group'] = pd.Categorical(
    np.select([pct_used <= 0, pct_used < users_median], ['Non-user', 'Lower user'], 'Higher user'),
    categories=['Non-user', 'Lower user', 'Higher user'], ordered=True)
print(f'Median % used among users: {users_median:.0%}')

ranked_cols = [key, 'Usage group', '% Used (MF + Promo)', 'MF % Used', 'Promo % Used', 'Marketing Fund Budget', 'Tier (used)',
               'Sales 3yr Avg', 'RSM', 'total Portal Logins YTD', 'Highspot Pitches Sent', 'BuyGreatDane.com page', '# Components used']
net.sort_values('% Used (MF + Promo)', ascending=False)[ranked_cols].style.format(
    {'% Used (MF + Promo)': '{:.0%}', 'MF % Used': '{:.0%}', 'Promo % Used': '{:.0%}',
     'Marketing Fund Budget': '${:,.0f}', 'Sales 3yr Avg': '${:,.0f}', 'Tier (used)': '{:.0f}'}, na_rep='—')

In [ ]:
profile = net.groupby('Usage group', observed=True).agg(**{
    'Dealers': (key, 'size'),
    'Avg % used': ('% Used (MF + Promo)', 'mean'),
    'Avg tier': ('Tier (used)', 'mean'),
    'Median 3yr sales': ('Sales 3yr Avg', 'median'),
    'Avg allocation': ('Marketing Fund Budget', 'mean'),
    'Avg dealer score': ('Dealer Score', 'mean'),
    'Avg portal logins': ('total Portal Logins YTD', 'mean'),
    '% with portal logins': ('Dealer Portal', 'mean'),
    'Avg Highspot pitches': ('Highspot Pitches Sent', 'mean'),
    '% using Highspot': ('Highspot', 'mean'),
    '% BuyGreatDane page active': ('BuyGreatDane.com page', 'mean'),
    'Avg engagement score': ('Overall Engagement Score', 'mean'),
    'Avg components used': ('# Components used', 'mean'),
}).T
profile

In [ ]:
# RSM and channel mix by usage group (counts)
display(pd.crosstab(net['RSM'].fillna('(none)'), net['Usage group']))
pd.crosstab(net['Channel Type'], net['Usage group'])

## 11. Usage by RSM or region

⚠️ **Region:** no region or territory field exists in the exports, so only RSM is analysed.
To add region, fill `DEALER_REGION` (dealer → region) or add a Region field to the Dealers grid; the cell picks it up automatically.

The RSM view covers **all** dealers, so corporate branches are included in the program-component adoption columns. The allocation columns only cover dealers with an allocation.

In [ ]:
DEALER_REGION = {}   # e.g. {'Atlantic Great Dane': 'Northeast', ...}

def rsm_view(group_col, frame_all, frame_alloc):
    alloc_part = usage_by(frame_alloc, group_col)[['Dealers', 'Allocation', 'Spend', 'Pooled % used', 'Avg dealer % used', '% using any fund']]
    alloc_part = alloc_part.rename(columns={'Dealers': 'Dealers w/ allocation'})
    g = frame_all.groupby(group_col, dropna=False)
    adopt = pd.DataFrame({'All dealers': g.size(),
                          '% any program component': g['Any program component'].mean(),
                          '% Highspot': g['Highspot'].mean(),
                          '% Portal': g['Dealer Portal'].mean()})
    return adopt.join(alloc_part, how='left')

rsm_all = used.merge(dealers[[key, 'RSM']], on=key, how='left')
rsm_all['RSM'] = rsm_all['RSM'].fillna('(no RSM)')
net['RSM'] = net['RSM'].fillna('(no RSM)')
rsm_fmt = usage_fmt | {'% any program component': '{:.0%}', '% Highspot': '{:.0%}', '% Portal': '{:.0%}', 'Dealers w/ allocation': '{:.0f}'}

display(rsm_view('RSM', rsm_all, net).style.format(rsm_fmt, na_rep='—'))

if DEALER_REGION:
    rsm_all['Region'] = rsm_all[key].map(DEALER_REGION).fillna('(unmapped)')
    net['Region'] = net[key].map(DEALER_REGION).fillna('(unmapped)')
    display(rsm_view('Region', rsm_all, net).style.format(rsm_fmt, na_rep='—'))
else:
    print('DEALER_REGION is empty; region view skipped.')

## 12. RSM tenure vs. dealer usage

⛔ **Can't be answered from the current exports:** no RSM start or hire dates are included.
Fill `RSM_START_DATE` below (RSM → date they took the role or territory) and the cell will compute tenure and relate it to their dealers' usage.
With 6–7 RSMs, treat any correlation as directional, not statistically meaningful.

In [ ]:
RSM_START_DATE = {}   # e.g. {'Steve Strickland': '2019-04-01', 'George Haas': '2022-01-15', ...}

if RSM_START_DATE:
    by_rsm = rsm_view('RSM', rsm_all, net).reset_index()
    by_rsm['Tenure (yrs)'] = by_rsm['RSM'].map(
        lambda r: (AS_OF - pd.Timestamp(RSM_START_DATE[r])).days / 365.25 if r in RSM_START_DATE else np.nan)
    missing = sorted(set(by_rsm['RSM']) - set(RSM_START_DATE) - {'(no RSM)'})
    if missing:
        print('No start date for:', missing)
    display(by_rsm.sort_values('Tenure (yrs)'))
    for metric in ['Pooled % used', '% using any fund', '% any program component']:
        d = by_rsm.dropna(subset=['Tenure (yrs)', metric])
        print(f'Spearman tenure vs {metric}: {spearman(d["Tenure (yrs)"], d[metric]):.2f} (n={len(d)} RSMs)')
else:
    print('RSM_START_DATE is empty; see the note above.')

# Revenue Touched per Support Dollar

## 13. Current figure: overall, by fund type and by activity type

Definitions (all $ from the Airtable rollups):
- **Airtable definition**: `Total Revenue Touched / Total Marketing Support Spend`, which is the existing *Revenue Touched Multiple* field.
  ⚠️ It counts **Highspot** revenue touched in the numerator, but Highspot has no $ in the denominator. This inflates the figure.
- **Funded-only**: (Promo + Reimbursement revenue touched) / (Promo + MF + Growth + Signage spend), which leaves Highspot out.
- **By fund**: each fund's revenue touched / that fund's spend. Signage has no revenue-touched field.
- Pipeline touched per $ is shown alongside as a leading indicator.

⚠️ Points to confirm with whoever owns the Airtable rollups (also flagged in section 2):
1. **Attribution:** whether one opportunity can count toward several programs (e.g. Highspot and Promo for the same deal). If so, totals double-count.
2. **Timing:** opportunities count as "touched" even when the pitch was sent after the deal closed (see `Pitch Timing` in the opps export).
3. Reimbursement revenue touched doesn't always equal MF + Growth revenue touched; see the reconciliation check.

In [ ]:
r = rt.merge(recon[[key, 'Marketing Fund Spend', 'Growth Fund Spend', 'Signage Spend', 'Promo Wallet Spend']], on=key, how='left') \
      .merge(dealers[[key, 'Promo Pipeline Touched', 'Highspot Pipeline Touched', 'Total Pipeline Touched']], on=key, how='left').fillna(0)

def ratio(num, den):
    return num / den if den else np.nan

rows = [
    ('Overall (Airtable definition)', r['Total Revenue Touched'].sum(), r['Total Pipeline Touched'].sum(), r['Total Marketing Support Spend'].sum()),
    ('Overall (funded-only, excl. Highspot)',
        r['Promo Revenue Touched'].sum() + r['Reimbursement Revenue Touched'].sum(),
        r['Promo Pipeline Touched'].sum() + r['Reimbursement Pipeline Touched'].sum(),
        r[['Promo Wallet Spend', 'Marketing Fund Spend', 'Growth Fund Spend', 'Signage Spend']].sum().sum()),
    ('Marketing Fund', r['Marketing Fund Revenue Touched'].sum(), r['Marketing Fund Pipeline Touched'].sum(), r['Marketing Fund Spend'].sum()),
    ('Growth Fund',    r['Growth Fund Revenue Touched'].sum(),    r['Growth Fund Pipeline Touched'].sum(),    r['Growth Fund Spend'].sum()),
    ('Promo Wallet',   r['Promo Revenue Touched'].sum(),          r['Promo Pipeline Touched'].sum(),          r['Promo Wallet Spend'].sum()),
    ('Reimbursement (MF + Growth + Signage)', r['Reimbursement Revenue Touched'].sum(), r['Reimbursement Pipeline Touched'].sum(),
        r[['Marketing Fund Spend', 'Growth Fund Spend', 'Signage Spend']].sum().sum()),
    ('Signage', np.nan, np.nan, r['Signage Spend'].sum()),
]
rtpd = pd.DataFrame(rows, columns=['Scope', 'Revenue Touched', 'Pipeline Touched', 'Support Spend']).set_index('Scope')
rtpd['Revenue Touched per $'] = [ratio(a, b) for a, b in zip(rtpd['Revenue Touched'], rtpd['Support Spend'])]
rtpd['Pipeline Touched per $'] = [ratio(a, b) for a, b in zip(rtpd['Pipeline Touched'], rtpd['Support Spend'])]
rtpd['Dealers w/ revenue touched'] = [
    (r['Total Revenue Touched'] > 0).sum(), ((r['Promo Revenue Touched'] + r['Reimbursement Revenue Touched']) > 0).sum(),
    (r['Marketing Fund Revenue Touched'] > 0).sum(), (r['Growth Fund Revenue Touched'] > 0).sum(),
    (r['Promo Revenue Touched'] > 0).sum(), (r['Reimbursement Revenue Touched'] > 0).sum(), np.nan]

rtpd.style.format({'Revenue Touched': '${:,.0f}', 'Pipeline Touched': '${:,.0f}', 'Support Spend': '${:,.0f}',
                   'Revenue Touched per $': '${:,.2f}', 'Pipeline Touched per $': '${:,.2f}',
                   'Dealers w/ revenue touched': '{:.0f}'}, na_rep='—')

In [ ]:
# Sanity check: recompute Airtable's Revenue Touched Multiple per dealer
chk = r[r['Total Marketing Support Spend'] > 0].copy()
chk['multiple_calc'] = chk['Total Revenue Touched'] / chk['Total Marketing Support Spend']
chk['matches Airtable'] = (chk['multiple_calc'] - chk['Revenue Touched Multiple']).abs() < 0.01
chk[[key, 'Total Revenue Touched', 'Total Marketing Support Spend', 'Revenue Touched Multiple', 'multiple_calc', 'matches Airtable']]

In [ ]:
# By activity type: only requests whose dealer has one request in that fund carry an exact revenue figure
# (same constraint as section 7). Coverage shows how much of each activity type is measurable.
req_rt = requests.copy()
single = req_rt['Requests for dealer in fund'] == 1
req_rt['Revenue touched'] = req_rt['Dealer fund revenue touched'].where(single)
req_rt['Pipeline touched'] = req_rt['Dealer fund pipeline touched'].where(single)

act = req_rt.groupby('Activity Type').agg(
    Requests=('Request', 'size'),
    Measurable=('Revenue touched', 'count'),
    Spend=('Amount', 'sum'),
    Revenue_touched=('Revenue touched', 'sum'),
    Pipeline_touched=('Pipeline touched', 'sum'))
act['Revenue touched per $'] = act['Revenue_touched'] / act['Spend'].where(act['Spend'] > 0)
act['Pipeline touched per $'] = act['Pipeline_touched'] / act['Spend'].where(act['Spend'] > 0)
act.sort_values('Revenue touched per $', ascending=False).style.format(
    {'Spend': '${:,.0f}', 'Revenue_touched': '${:,.0f}', 'Pipeline_touched': '${:,.0f}',
     'Revenue touched per $': '${:,.2f}', 'Pipeline touched per $': '${:,.2f}'}, na_rep='—')

## 14. Highest and lowest Revenue Touched per Support Dollar: dealers and activities

Only dealers with support spend > $0 have a defined figure. Dealers with $0 revenue touched are kept in, because they're the "lowest" and that matters.

In [ ]:
dealer_rt = r[r['Total Marketing Support Spend'] > 0][[key, 'Total Marketing Support Spend', 'Total Revenue Touched',
                                                      'Highspot Revenue Touched', 'Promo Revenue Touched', 'Reimbursement Revenue Touched',
                                                      'Total Pipeline Touched']].copy()
dealer_rt['Funded revenue touched'] = dealer_rt['Promo Revenue Touched'] + dealer_rt['Reimbursement Revenue Touched']
dealer_rt['Rev per $ (Airtable)'] = dealer_rt['Total Revenue Touched'] / dealer_rt['Total Marketing Support Spend']
dealer_rt['Rev per $ (funded-only)'] = dealer_rt['Funded revenue touched'] / dealer_rt['Total Marketing Support Spend']
dealer_rt['Pipeline per $'] = dealer_rt['Total Pipeline Touched'] / dealer_rt['Total Marketing Support Spend']
dealer_rt = dealer_rt.merge(dealer_ref[[key, 'Tier (used)', 'RSM', 'Channel Type']], on=key, how='left')

money = {c: '${:,.0f}' for c in ['Total Marketing Support Spend', 'Total Revenue Touched', 'Highspot Revenue Touched',
                                 'Promo Revenue Touched', 'Reimbursement Revenue Touched', 'Funded revenue touched', 'Total Pipeline Touched']}
dealer_rt.sort_values('Rev per $ (funded-only)', ascending=False).style.format(
    money | {'Rev per $ (Airtable)': '${:,.2f}', 'Rev per $ (funded-only)': '${:,.2f}', 'Pipeline per $': '${:,.2f}', 'Tier (used)': '{:.0f}'},
    na_rep='—')

In [ ]:
# Per-request view (activities), measurable requests only
req_rt['Revenue touched per $'] = req_rt['Revenue touched'] / req_rt['Amount'].where(req_rt['Amount'] > 0)
req_rt.dropna(subset=['Revenue touched per $']).sort_values('Revenue touched per $', ascending=False)[
    [key, 'Fund', 'Request', 'Activity Type', 'Amount', 'Revenue touched', 'Pipeline touched', 'Revenue touched per $']
].style.format({'Amount': '${:,.0f}', 'Revenue touched': '${:,.0f}', 'Pipeline touched': '${:,.0f}', 'Revenue touched per $': '${:,.2f}'})

In [ ]:
plot = dealer_rt.set_index(key)['Rev per $ (funded-only)'].sort_values()
fig, ax = plt.subplots(figsize=(7, max(2.5, 0.35 * len(plot))))
ax.barh(plot.index, plot.values, color='#4C72B0')
ax.set_xscale('symlog', linthresh=1)
ax.set_xlabel('Funded revenue touched per support $ (symlog)')
ax.set_title('Revenue Touched per Support Dollar by dealer')
for i, v in enumerate(plot.values):
    ax.text(v, i, f' ${v:,.0f}', va='center', fontsize=8)
plt.tight_layout()